In [1]:
#------------------------------------------------------------------------------------------------------#
#
# Code:      CC01_CCAR_C05_AgenticAI_model_scoring_03.ipynb
#
# Objective: Step C05: Use AutoGen to develop an more advanced version of Agentic AI to score vintages of 201605, 201606, 201607, ..., 201801.
#                      It is level-3 Agentic AI.
#                      _03.ipynb is to output all statistics of all vintages into one summary table.
#
#            Jingru Chen
#            2026-03-23
#
#----------------------------------------------------------------------------------------------------#

In [2]:
# This version moves from a single Class-based Agent to AutoGen, which is the jump from a "Single Actor" to a "Multi-Agent Orchestration."

# In AutoGen, I don't just have one class managing everything. I create a ConversableAgent (the Analyst) and a UserProxyAgent (the Executor/Manager).
# This is significantly more "Agentic" because the agents can negotiate—for example, the Analyst can see the drift, and instead of just stopping,
# it can "chat" with a Coder agent to try a different threshold or a different model version before giving up.

In [3]:
### Level       Type                                   Financial Risk Example (CCAR)
### Level-3    Multi-Agent Orchestration (AutoGen)     A "Manager Agent" assigns a "Coder Agent" to fix a drift issue, while a "Reviewer Agent" audits the fix.

# Step 0: Upload libraries

In [4]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

from pydantic import BaseModel, Field

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import os
if os.path.exists('/content/drive/MyDrive'):
    print("✅ Drive is mounted. Root folders:")
    print(os.listdir('/content/drive/MyDrive'))
else:
    print("❌ Drive not found. Run the mount command below.")

✅ Drive is mounted. Root folders:
['1st Manuscript', '_PDF_001.pdf', 'Proper receiver operating characteristic analysis - The bigamma model.pdf', 'Proper Binormal ROC Curves - Theory and Maximum-Likelihood Estimation.pdf', 'Continuous versus Categorical Data for ROC Analysis - Some Quantitative Considerations.pdf', 'Receiver Operating Characteristic Analysis - A Tool for the Quantitative Evaluation of Observer Performance and Imaging Systems.pdf', 'The Proper binormal model - Parametric receiver operating characteristic curve estimation with degenerate data.pdf', 'ROC Papers', 'Proposal_Paper_v110.pdf', 'Comparison of Receiver Operating Characteristic.pdf', '搜索 (3).gdoc', '搜索 (2).gdoc', '搜索 (1).gdoc', '搜索.gdoc', 'Photos', 'Jingru (1).zip', '2001-02 (1).JPG', '2001-04 (1).JPG', 'Jingru.zip', '2001-01 (1).JPG', '2001-03 (1).JPG', '2001-07.JPG', '2001-10.JPG', '2001-11.JPG', '2001-06.JPG', '2006-02.JPG', '2005-02.JPG', '2001-12.JPG', '2001-04.JPG', '2008-06.JPG', '2008-02.JPG', '2006-05.J

In [7]:
!ls "/content/drive/MyDrive/Colab Notebooks/" | grep ".pkl"

ccar_pd_model_2026-03-22.pkl


In [8]:
ls -ltr

total 8
drwxr-xr-x 1 root root 4096 Mar 23 20:21 sample_data/
drwx------ 5 root root 4096 Mar 23 20:24 drive/


In [10]:
import sys
from google.colab import drive

# 1. Mount the drive
drive.mount('/content/drive')

# 2. Add the specific folder to Python's search path
# Note the space in "Colab Notebooks"
path_to_config = '/content/drive/MyDrive/Colab Notebooks'
if path_to_config not in sys.path:
    sys.path.append(path_to_config)

# 3. Now you can import normally
from config import MY_API_KEY

print(f"Success! Key loaded: {MY_API_KEY[:5]}****")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Success! Key loaded: sk-pr****


In [11]:
from datetime import datetime
from zoneinfo import ZoneInfo

run_date="2026-03-23"

start = datetime.now( ZoneInfo("America/New_York"))

print( start.strftime("%Y-%m-%d %H:%M:%S %Z"))     # 2026-03-17 17:34:58 EDT
print( start.strftime("%Y-%m-%d %I:%M:%S %p %Z"))  # 2026-03-17 05:34:58 PM EDT

2026-03-23 17:26:17 EDT
2026-03-23 05:26:17 PM EDT


In [12]:
!pip install -U "autogen[gemini]"

In [13]:
import autogen
from autogen import AssistantAgent, UserProxyAgent
from google.colab import userdata

In [14]:
x_list=['original_balance', 'credit_score_orig', 'loan_to_value_orig', 'interest_rate', 'loan_term_months',
        'delta_Unemployment1',
       'delta_Mortgage_rate1', 'delta_House_Price_Index__Level1',
       'delta_Unemployment3', 'delta_Mortgage_rate3',
       'delta_House_Price_Index__Level3', 'delta_Unemployment6',
       'delta_Mortgage_rate6', 'delta_House_Price_Index__Level6',
       'delta_Unemployment12', 'delta_Mortgage_rate12',
       'delta_House_Price_Index__Level12', 'delta_Unemployment24',
       'delta_Mortgage_rate24', 'delta_House_Price_Index__Level24']

x_list_v2=['original_balance', 'credit_score_orig', 'loan_to_value_orig', 'interest_rate',
        'delta_Unemployment1',
       'delta_Unemployment3',
       'delta_Unemployment6',
       'delta_Mortgage_rate6', 'delta_House_Price_Index__Level6',
       'delta_Unemployment12',
       'delta_Unemployment24' ]

# y_list= ['flag_default']

pd_model = 'ccar_pd_model_2026-03-22.pkl'

# pd_input_file= "/CCAR_Mortgage_data_for_model_DEV_20260319_01.csv"
# Use the full Absolute Path:
pd_input_file = "/content/drive/MyDrive/Colab Notebooks/CCAR_Mortgage_data_for_model_DEV_20260319_01.csv"

# pd_vintage = 201604
pd_x_list = x_list_v2
pd_y = 'flag_default'
pd_threshold = 0.50      ### pd_threshold value will be used in the agent_a step.


In [15]:
class CCARProductionAgent:
    """
    An Autonomous Agentic AI for CCAR Production.
    Features: Persistent State, Decision Layer, and Drift Protection.
    """
    def __init__(self, agent_name, model_path, input_file, x_list, y_target, threshold, output_path="./"):
        self.agent_name = agent_name
        self.model = joblib.load(model_path)
        self.input_file = input_file
        self.x_list = x_list
        self.y_target = y_target
        self.threshold = threshold
        self.output_path = output_path

        # --- AGENT MEMORY ---
        self.recall_history = []
        self.master_report = None
        self.is_aborted = False

    def _check_model_drift(self, current_recall, vintage):
        """
        DECISION LAYER: Evaluates if the Agent should continue or stop.
        Logic: If Recall < 0.60 for 3 consecutive runs, trigger a Kill Switch.
        This is the "brain" of the agent. It evaluates its own performance against a regulatory standard.
        """
        self.recall_history.append(current_recall)

        # Look at the last 3 performance observations
        if len(self.recall_history) >= 3:
            recent_performance = self.recall_history[-3:]
            if all(r < 0.60 for r in recent_performance):
                self.is_aborted = True
                print(f"\n [CRITICAL ALERT - {self.agent_name}]")
                print(f"Model Drift Detected at vintage {vintage}!")
                print(f"Recall has dropped below 60% for 3 consecutive months: {recent_performance}")
                print("ABORTING PRODUCTION RUN TO PREVENT REGULATORY MISREPORTING.")

    def _scoring_action(self, vintage: str):
        if self.is_aborted:
            return {"vintage": vintage, "status": "ABORTED (Drift Detected)"}

        try:
            # FIX: Use os.path.join to combine the output path and input file
            import os
            full_path = os.path.join(self.output_path, self.input_file)

            # Diagnostic print (Helps you see exactly what the Agent sees)
            # print(f"--- DEBUG: Attempting to open: {full_path} ---")

            df = pd.read_csv(full_path)

            # df = pd.read_csv(self.output_path + self.input_file)

            df_v = df.loc[df['report_yrmo'].astype(str) == str(vintage)].reset_index(drop=True)

            if df_v.empty:
                return {"vintage": vintage, "status": "Skipped (No Data)"}

            # Inference
            X, y = df_v[self.x_list], df_v[self.y_target]
            y_proba = self.model.predict_proba(X)[:, 1]
            y_pred = (y_proba >= self.threshold).astype(int)

            # Metrics
            tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
            rec = recall_score( y, y_pred, zero_division=0)
            precision= precision_score( y, y_pred, zero_division=0)

            # --- AGENT DECISION STEP ---
            self._check_model_drift(rec, vintage)

            return {
                "vintage": vintage,
                "status": "Success" if not self.is_aborted else "ABORTED",
                "count": len(df_v),
                "avg_pd": y_proba.mean(),
                "precision": precision,
                "recall": rec,
                "f1": f1_score(y, y_pred, zero_division=0),
                "TP": tp, "FN": fn, "TN": tn, "FP": fp
            }
        except Exception as e:
            return {"vintage": vintage, "status": f"Error: {str(e)}"}

    def run_production(self, start_date, end_date):
        vintages = pd.date_range(start=pd.to_datetime(start_date, format='%Y%m'),
                                 end=pd.to_datetime(end_date, format='%Y%m'),
                                 freq='MS').strftime('%Y%m').tolist()

        print(f" [{self.agent_name}] Starting production for {len(vintages)} vintages...")

        results = []
        for v in vintages:
            if self.is_aborted: break
            results.append(self._scoring_action(v))

        self.master_report = pd.DataFrame(results)
        return self.master_report

# Step 1: The AutoGen "Conversation" Architecture

In [16]:
import autogen
from autogen import AssistantAgent, UserProxyAgent

# 1. Ensure the environment variable is set in the current session
import os
os.environ["OPENAI_API_KEY"] = MY_API_KEY

# 2. Define the config list for the Agents
config_list = [
    {
        "model": "gpt-4", # or "gpt-3.5-turbo"
        "api_key": os.environ.get("OPENAI_API_KEY")
    }
]

# 3. Apply it to the Agents
llm_config = {"config_list": config_list, "cache_seed": 42}

analyst = AssistantAgent(
    name="CCAR_Analyst",
    llm_config=llm_config,
    system_message="You are a risk expert. Analyze CCAR mortgage vintage recall scores."
)

# 4. The 'Executor' (The Hands)
# # This agent actually runs your Python class methods.
# executor = UserProxyAgent(
#     name="Executor",
#     human_input_mode="NEVER",
#     max_consecutive_auto_reply=10,
#     is_termination_msg=lambda x: x.get("content", "").rstrip().endswith("TERMINATE"),
#     code_execution_config={"work_dir": "ccar_production", "use_docker": False}
# )

# Updated, safer termination logic
executor = UserProxyAgent(
    name="Executor",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=10,
    # The (x.get("content") or "") ensures we never call rstrip on None
    is_termination_msg=lambda x: (x.get("content") or "").rstrip().endswith("TERMINATE"),
    code_execution_config={"work_dir": "ccar_production", "use_docker": False}
)

# Step 2: Registering CCARProductionAgent Class as a "Tool"


##### In AutoGen, my CCARProductionAgent class becomes a Tool that the Executor can use. The Analyst doesn't see the code; it only sees the results (the metrics) and decides the next move.

In [17]:
# 1. Create your agent instance first (The "Physical" Agent)
# agent_a = CCARProductionAgent("Agent_Conservative", pd_model, pd_input_file, pd_x_list, pd_y, 0.05)

# Define the full path to your model in Google Drive
# Make sure the filename matches exactly what you see in your Drive folder
pd_model_path = "/content/drive/MyDrive/Colab Notebooks/ccar_pd_model_2026-03-22.pkl"

# Now create the agent using the full path
agent_a = CCARProductionAgent(
    agent_name="Agent_Conservative",
    model_path=pd_model_path,
    input_file=pd_input_file, # This now has the full /content/drive/... path
    x_list=pd_x_list,
    y_target=pd_y,
    threshold=pd_threshold,   # pd_threshold = 0.15
    output_path="" # Set this to empty so it doesn't prefix the path
)


# 2. Register a "Wrapper" function that points to that agent
@executor.register_for_execution()
@analyst.register_for_llm(description="Runs PD scoring for a specific vintage.")
def run_vintage_production(vintage: str) -> str:
    result = agent_a._scoring_action(vintage)

    # Update the master_report DataFrame inside the agent every time a month is run
    new_row = pd.DataFrame([result])
    if agent_a.master_report is None:
        agent_a.master_report = new_row
    else:
        # Append the new result to the existing report
        agent_a.master_report = pd.concat([agent_a.master_report, new_row], ignore_index=True)

    return str(result)


# 3. Register a tool to retrieve the final summary table
@executor.register_for_execution()
@analyst.register_for_llm(description="Retrieves and prints the final aggregated CCAR production report.")
def get_final_report() -> str:
    """
    Accesses the master_report in the physical agent and returns it as a string.
    """
    if agent_a.master_report is not None:
        # Return the string representation of the DataFrame for the Analyst to display
        return agent_a.master_report.to_string()
    else:
        # If the Analyst calls this before scoring, return a warning
        return "No report data available yet. Please run scoring first."


# Step 3. The Agentic Interaction

In [18]:
message = """
1. Score mortgage vintages from 201605 to 201612 using 'run_vintage_production'.
2. After each month, analyze the Recall.
3. If you see a downward trend or Recall < 0.60, explain why and stop.
4. ONCE FINISHED (or if aborted), call 'get_final_report' and print the entire table clearly.
"""

executor.initiate_chat(analyst, message=message)

Executor (to CCAR_Analyst):


1. Score mortgage vintages from 201605 to 201612 using 'run_vintage_production'.
2. After each month, analyze the Recall.
3. If you see a downward trend or Recall < 0.60, explain why and stop.
4. ONCE FINISHED (or if aborted), call 'get_final_report' and print the entire table clearly.


--------------------------------------------------------------------------------
CCAR_Analyst (to Executor):

***** Suggested tool call (call_MDcYUdMHPVw4rjEa1Jemd9Jy): run_vintage_production *****
Arguments: 
{
  "vintage": "201605"
}
***************************************************************************************

--------------------------------------------------------------------------------

>>>>>>>> EXECUTING FUNCTION run_vintage_production...
Call ID: call_MDcYUdMHPVw4rjEa1Jemd9Jy
Input arguments: {'vintage': '201605'}

>>>>>>>> EXECUTED FUNCTION run_vintage_production...
Call ID: call_MDcYUdMHPVw4rjEa1Jemd9Jy
Input arguments: {'vintage': '201605'}
Output:
{'

ChatResult(chat_id=75947320893886395322225914918839490518, chat_history=[{'content': "\n1. Score mortgage vintages from 201605 to 201612 using 'run_vintage_production'.\n2. After each month, analyze the Recall.\n3. If you see a downward trend or Recall < 0.60, explain why and stop.\n4. ONCE FINISHED (or if aborted), call 'get_final_report' and print the entire table clearly.\n", 'role': 'assistant', 'name': 'Executor'}, {'tool_calls': [{'id': 'call_MDcYUdMHPVw4rjEa1Jemd9Jy', 'function': {'arguments': '{\n  "vintage": "201605"\n}', 'name': 'run_vintage_production'}, 'type': 'function'}], 'content': None, 'role': 'assistant'}, {'content': "{'vintage': '201605', 'status': 'Success', 'count': 1157, 'avg_pd': np.float64(0.30779203325949267), 'precision': 0.0641025641025641, 'recall': 0.19230769230769232, 'f1': 0.09615384615384616, 'TP': np.int64(5), 'FN': np.int64(21), 'TN': np.int64(1058), 'FP': np.int64(73)}", 'tool_responses': [{'tool_call_id': 'call_MDcYUdMHPVw4rjEa1Jemd9Jy', 'role': 't

In [19]:
from datetime import datetime
end = datetime.now(ZoneInfo("America/New_York"))
duration = end - start

print(f"Started:  {start}")
print(f"Finished: {end}")
print(f"\nDuration: {duration}")                    # 0:00:02.351234
print(f"Duration: {duration.total_seconds():.3f} seconds")

Started:  2026-03-23 17:26:17.408025-04:00
Finished: 2026-03-23 17:28:44.476061-04:00

Duration: 0:02:27.068036
Duration: 147.068 seconds
